# 00 — Kaggle ortam ve deney kapısı kontrolü

**Amaç:** Kaggle GPU/Python paket sürümlerini kaydetmek ve mevcut Stage 3 kaynak paketindeki salt-okunur hazır olma kontrolünü çalıştırmak.

**Bu notebook model yüklemez, model yanıtı üretmez ve eğitim yapmaz.** Kontrol engelleri beklenen çıktıdır; engelleri gizlemek veya inference iznini açmak için manifest değiştirilmez.

## Çalıştırmadan önce
1. Kaggle Notebook ayarlarında Accelerator = **GPU** seç.
2. İnternet erişimini aç (Kaggle hesabında izin veriliyorsa).
3. Daha önce yüklenmiş `dijitalikiz-stage3-source` veri kümesini notebook'a Input olarak ekle.
4. Hücreleri sırayla çalıştır ve son hücredeki raporu sakla.

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import platform
import subprocess
import sys
import hashlib
from datetime import datetime, timezone

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

runtime = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": package_version("torch"),
    "transformers": package_version("transformers"),
    "accelerate": package_version("accelerate"),
}
try:
    import torch
    runtime["torch_cuda_version"] = torch.version.cuda
    runtime["cuda_available"] = bool(torch.cuda.is_available())
    runtime["gpu_count"] = torch.cuda.device_count()
    runtime["gpu_names"] = [
        torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())
    ] if torch.cuda.is_available() else []
except Exception as exc:
    runtime["torch_import_error"] = f"{type(exc).__name__}: {exc}"

print(json.dumps(runtime, indent=2, ensure_ascii=False))
Path("/kaggle/working").mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/kaggle_environment_probe.json").write_text(
    json.dumps(runtime, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)


In [ ]:
# Kaynak veri kümesini otomatik bul; birden fazla kopya varsa sessizce seçim yapma.
input_root = Path("/kaggle/input")
manifest_hits = list(input_root.rglob("EXPERIMENT_MANIFEST_STAGE3.json")) if input_root.exists() else []
print("Manifest adayları:")
for p in manifest_hits:
    print(" -", p)

if len(manifest_hits) != 1:
    raise RuntimeError(
        f"Tam olarak 1 EXPERIMENT_MANIFEST_STAGE3.json bekleniyordu; bulunan: {len(manifest_hits)}. "
        "Kaggle Input bölümüne doğru Stage 3 kaynak veri kümesini ekle ve yinelenen kopyaları kaldır."
    )

manifest_path = manifest_hits[0]
repo_root = manifest_path.parents[2]
required_paths = [
    repo_root / "scripts" / "evidence_abstention_preflight.py",
    repo_root / "research_candidates" / "evidence_abstention" / "CASE_REGISTRY.json",
    repo_root / "research_candidates" / "evidence_abstention" / "RUNTIME_LOCK.json",
    repo_root / "research_candidates" / "evidence_abstention" / "OUTPUT_SCHEMA.json",
]
missing = [str(p) for p in required_paths if not p.is_file()]
if missing:
    raise FileNotFoundError("Kaynak paket eksik dosyalar içeriyor:\n" + "\n".join(missing))

print("Kaynak kökü:", repo_root)
print("Manifest SHA-256:", hashlib.sha256(manifest_path.read_bytes()).hexdigest())
print("Kaynak kökü doğrulandı.")


In [ ]:
# Kanonik salt-okunur preflight: model yüklemez ve inference çalıştırmaz.
preflight_script = repo_root / "scripts" / "evidence_abstention_preflight.py"
proc = subprocess.run(
    [sys.executable, str(preflight_script), "--root", str(repo_root)],
    capture_output=True,
    text=True,
    check=False,
)
print("Preflight process exit code:", proc.returncode)
if proc.stdout:
    print(proc.stdout)
if proc.stderr:
    print("STDERR:")
    print(proc.stderr)

try:
    preflight_report = json.loads(proc.stdout)
except json.JSONDecodeError as exc:
    raise RuntimeError("Preflight geçerli JSON raporu üretmedi.") from exc

print("\nÖZET")
print("Hazır mı?:", preflight_report.get("ready"))
print("Kabul edilmiş vaka:", preflight_report.get("accepted_case_count"))
print("Kabul edilmiş proje:", preflight_report.get("accepted_project_count"))
print("Engel sayısı:", preflight_report.get("blocker_count"))
print("Model inference yapıldı mı?:", preflight_report.get("model_inference_performed"))

report_path = Path("/kaggle/working/stage3_read_only_preflight_report.json")
report_path.write_text(json.dumps(preflight_report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("Rapor kaydedildi:", report_path)
print("ÖNEMLİ: ready=False ise bu beklenen bir blokaj raporudur; inference başlatma.")


## Sonraki adım

Bu notebook **hazır olma kapısını** kontrol eder. `ready=False` çıkması notebook hatası değildir; eksik bilimsel/teknik koşulları listeler. Raporu ve `kaggle_environment_probe.json` dosyasını sakla.

Modeli yalnızca yükleyip ağırlık/revision/runtime doğrulayan ikinci notebook'a geçiş için önce ortam sürümlerini ve bu rapordaki engelleri gözden geçireceğiz. Bu aşamada model yanıtı üretme veya eğitim yok.